[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.5_multimodal_serving/lab.ipynb) [![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/marimo-badge.svg)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.5_multimodal_serving/lab.ipynb)

# Lab 5.5: Multimodal Serving Analysis

This lab explores the compute and scheduling characteristics of multimodal serving architectures through analytical simulations.

In [ ]:
# Install dependencies (numpy for computation, matplotlib for visualization)
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
# Core imports for all experiments
import numpy as np  # numerical computation
import matplotlib.pyplot as plt  # visualization
import matplotlib.patches as mpatches  # legend patches for Gantt chart

## Experiment 1: Model Complexity Comparison

Compare compute graph sizes across text-only and multimodal architectures. Multimodal models have more stages, each with distinct compute profiles.

In [ ]:
def experiment_1_model_complexity():
    """Compare compute graph complexity: text-only vs multimodal models."""
    # Define model architectures and their component counts
    model_names = ['GPT-4\n(Text)', 'LLaVA\n(Vision+Text)', 'Qwen3-Omni\n(All)', 'Orpheus\n(Speech)', 'Sora\n(Video)']
    
    # Number of distinct compute stages per model
    n_stages = np.array([2, 3, 5, 4, 4])  # tokenize+decode, +encoder, +encoder+codec+diffusion, etc.
    
    # Total FLOPS per request (relative to text-only baseline)
    relative_flops = np.array([1.0, 1.8, 3.2, 2.4, 12.0])
    
    # Memory footprint multiplier vs text-only
    memory_mult = np.array([1.0, 1.4, 2.1, 1.6, 4.5])
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left plot: compute stages and relative FLOPS
    bar_width = 0.35  # width of each bar group
    x_positions = np.arange(len(model_names))  # x-axis positions for model groups
    
    # Plot stages as bars
    bars1 = ax1.bar(x_positions - bar_width/2, n_stages, bar_width,
                    label='Compute Stages', color='#dbeafe', edgecolor='#000')
    # Plot relative FLOPS as bars
    bars2 = ax1.bar(x_positions + bar_width/2, relative_flops, bar_width,
                    label='Relative FLOPS (vs text)', color='#dcfce7', edgecolor='#000')
    
    ax1.set_xlabel('Model Architecture')  # x-axis label
    ax1.set_ylabel('Count / Multiplier')  # y-axis label
    ax1.set_title('Compute Graph Complexity')  # plot title
    ax1.set_xticks(x_positions)  # set tick positions
    ax1.set_xticklabels(model_names, fontsize=9)  # set tick labels
    ax1.legend()  # show legend
    ax1.grid(axis='y', alpha=0.3)  # light horizontal grid
    
    # Right plot: memory multiplier
    colors_mem = ['#f3f4f6', '#dcfce7', '#dbeafe', '#f3e8ff', '#ffe4e6']
    bars3 = ax2.bar(x_positions, memory_mult, color=colors_mem, edgecolor='#000')
    ax2.set_xlabel('Model Architecture')  # x-axis label
    ax2.set_ylabel('Memory Multiplier (vs text-only)')  # y-axis label
    ax2.set_title('GPU Memory Footprint')  # plot title
    ax2.set_xticks(x_positions)  # set tick positions
    ax2.set_xticklabels(model_names, fontsize=9)  # set tick labels
    ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Text baseline')  # baseline reference
    ax2.legend()  # show legend
    ax2.grid(axis='y', alpha=0.3)  # light horizontal grid
    
    plt.tight_layout()  # prevent label overlap
    plt.show()  # render the figure
    
    # Print summary statistics
    print(f'Max stage count: {n_stages.max()} (Qwen3-Omni)')  # highest complexity
    print(f'Max FLOPS multiplier: {relative_flops.max()}x (Sora video)')  # most compute-intensive
    print(f'Max memory multiplier: {memory_mult.max()}x (Sora video)')  # most memory-hungry
    return n_stages, relative_flops, memory_mult

# Execute experiment 1
exp1_stages, exp1_flops, exp1_memory = experiment_1_model_complexity()

## Experiment 2: EPD Disaggregation Simulation

Simulate the latency breakdown across Encode, Prefill, and Decode stages for different request types. Compare monolithic (all on one GPU) vs disaggregated (specialized pools) scheduling.

In [ ]:
def experiment_2_epd_disaggregation():
    """Simulate EPD stage latencies: monolithic vs disaggregated serving."""
    # Request types with different stage profiles (ms)
    request_types = ['Text Only', 'Single Image', 'Multi-Image\n(4 imgs)', 'Audio Input\n(10s clip)', 'Image+Audio']
    
    # Encode latency per request type (ms) - vision/audio preprocessing
    encode_latency_mono = np.array([0, 45, 180, 120, 165])  # monolithic: encoder blocks pipeline
    encode_latency_disagg = np.array([0, 30, 90, 80, 110])  # disaggregated: batched on compute-opt GPU
    
    # Prefill latency (ms) - KV cache construction
    prefill_latency_mono = np.array([25, 40, 85, 55, 95])  # monolithic: competes with decoder memory
    prefill_latency_disagg = np.array([25, 35, 60, 45, 70])  # disaggregated: dedicated memory-opt pool
    
    # Decode latency for 100 output tokens (ms)
    decode_latency_mono = np.array([200, 200, 200, 200, 200])  # same across types (autoregressive)
    decode_latency_disagg = np.array([180, 180, 180, 180, 180])  # slightly faster: no resource contention
    
    # Transfer overhead for disaggregated (activation shipping between pools)
    transfer_overhead = np.array([0, 3, 6, 3, 6])  # ms: 1-3ms per hop, 2 hops for multimodal
    
    # Total latencies
    total_mono = encode_latency_mono + prefill_latency_mono + decode_latency_mono
    total_disagg = encode_latency_disagg + prefill_latency_disagg + decode_latency_disagg + transfer_overhead
    
    # Create stacked bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x_pos = np.arange(len(request_types))  # bar positions
    bar_w = 0.6  # bar width
    
    # Left: Monolithic serving stacked bars
    ax1.bar(x_pos, encode_latency_mono, bar_w, label='Encode', color='#dcfce7', edgecolor='#000')
    ax1.bar(x_pos, prefill_latency_mono, bar_w, bottom=encode_latency_mono,
            label='Prefill', color='#dbeafe', edgecolor='#000')
    ax1.bar(x_pos, decode_latency_mono, bar_w,
            bottom=encode_latency_mono + prefill_latency_mono,
            label='Decode', color='#fef3c7', edgecolor='#000')
    ax1.set_title('Monolithic: All Stages on One GPU')  # plot title
    ax1.set_xlabel('Request Type')  # x-axis label
    ax1.set_ylabel('Latency (ms)')  # y-axis label
    ax1.set_xticks(x_pos)  # set x ticks
    ax1.set_xticklabels(request_types, fontsize=8)  # request type labels
    ax1.legend()  # show legend
    ax1.grid(axis='y', alpha=0.3)  # grid
    
    # Right: Disaggregated serving stacked bars
    ax2.bar(x_pos, encode_latency_disagg, bar_w, label='Encode', color='#dcfce7', edgecolor='#000')
    ax2.bar(x_pos, prefill_latency_disagg, bar_w, bottom=encode_latency_disagg,
            label='Prefill', color='#dbeafe', edgecolor='#000')
    ax2.bar(x_pos, decode_latency_disagg, bar_w,
            bottom=encode_latency_disagg + prefill_latency_disagg,
            label='Decode', color='#fef3c7', edgecolor='#000')
    ax2.bar(x_pos, transfer_overhead, bar_w,
            bottom=encode_latency_disagg + prefill_latency_disagg + decode_latency_disagg,
            label='Transfer', color='#ffe4e6', edgecolor='#000')
    ax2.set_title('Disaggregated: EPD Pools')  # plot title
    ax2.set_xlabel('Request Type')  # x-axis label
    ax2.set_ylabel('Latency (ms)')  # y-axis label
    ax2.set_xticks(x_pos)  # set x ticks
    ax2.set_xticklabels(request_types, fontsize=8)  # request type labels
    ax2.legend()  # show legend
    ax2.grid(axis='y', alpha=0.3)  # grid
    
    plt.tight_layout()  # prevent overlap
    plt.show()  # render
    
    # Print latency reduction summary
    reduction_pct = (total_mono - total_disagg) / total_mono * 100  # percent improvement
    for i, req in enumerate(request_types):
        print(f'{req.replace(chr(10), " "):20s}: {total_mono[i]:3d}ms -> {total_disagg[i]:3d}ms ({reduction_pct[i]:.1f}% reduction)')
    return total_mono, total_disagg

# Execute experiment 2
exp2_mono, exp2_disagg = experiment_2_epd_disaggregation()

## Experiment 3: Streaming Pipeline Simulation

Simulate the Thinker-Talker-Codec overlap in VoxServe-style speech serving. Visualize how chunk policies affect time-to-first-audio and GPU utilization.

In [ ]:
def experiment_3_streaming_pipeline():
    """Simulate chunked streaming pipeline for speech LLM serving."""
    # Chunk policies to compare (chunk duration in ms)
    chunk_sizes_ms = [50, 100, 200, 500]  # different chunk durations
    total_audio_duration_ms = 2000  # 2 seconds of speech output
    
    # Processing times per stage per chunk (ms)
    thinker_time_per_chunk = 80  # LLM generates semantic tokens for one chunk
    talker_time_per_chunk = 20  # codec converts semantic tokens to audio codes
    vocoder_time_per_chunk = 10  # synthesize waveform from codes
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))  # 4 subplots for 4 chunk sizes
    axes_flat = axes.flatten()  # flatten for iteration
    
    # Colors for each stage
    colors_stages = {'Thinker': '#dbeafe', 'Talker': '#dcfce7', 'Vocoder': '#f3e8ff'}
    
    ttfa_results = []  # collect time-to-first-audio for each policy
    
    for idx, chunk_ms in enumerate(chunk_sizes_ms):
        ax = axes_flat[idx]  # current subplot
        n_chunks = total_audio_duration_ms // chunk_ms  # number of chunks to process
        
        # Scale processing time by chunk size (larger chunks = more work)
        scale_factor = chunk_ms / 200.0  # normalized to 200ms reference
        thinker_t = thinker_time_per_chunk * scale_factor  # scaled thinker time
        talker_t = talker_time_per_chunk * scale_factor  # scaled talker time
        vocoder_t = vocoder_time_per_chunk * scale_factor  # scaled vocoder time
        
        # Compute overlapped schedule: each stage starts when its input chunk is ready
        schedule = []  # list of (stage_name, start_time, duration, chunk_idx)
        
        max_chunks_to_show = min(n_chunks, 6)  # limit display for readability
        
        for c in range(max_chunks_to_show):
            # Thinker starts sequentially (must finish chunk c before c+1)
            thinker_start = c * thinker_t  # thinker processes chunks sequentially
            # Talker starts after thinker finishes this chunk
            talker_start = thinker_start + thinker_t  # depends on thinker output
            # Vocoder starts after talker finishes this chunk
            vocoder_start = talker_start + talker_t  # depends on talker output
            
            schedule.append(('Thinker', thinker_start, thinker_t, c))
            schedule.append(('Talker', talker_start, talker_t, c))
            schedule.append(('Vocoder', vocoder_start, vocoder_t, c))
        
        # Plot Gantt-style chart
        stage_y = {'Thinker': 2, 'Talker': 1, 'Vocoder': 0}  # y-positions for stages
        
        for stage_name, start, duration, chunk_idx in schedule:
            y = stage_y[stage_name]  # vertical position
            color = colors_stages[stage_name]  # stage color
            ax.barh(y, duration, left=start, height=0.6,
                    color=color, edgecolor='#000', linewidth=0.5)
            # Label chunk index inside bar if wide enough
            if duration > 15:
                ax.text(start + duration/2, y, f'C{chunk_idx}',
                        ha='center', va='center', fontsize=7)
        
        # Time-to-first-audio: when first vocoder chunk completes
        ttfa = thinker_t + talker_t + vocoder_t  # first chunk through full pipeline
        ttfa_results.append(ttfa)  # store for summary
        
        # Mark TTFA on plot
        ax.axvline(x=ttfa, color='red', linestyle='--', alpha=0.7, label=f'TTFA={ttfa:.0f}ms')
        
        ax.set_yticks([0, 1, 2])  # y-axis ticks
        ax.set_yticklabels(['Vocoder', 'Talker', 'Thinker'])  # stage labels
        ax.set_xlabel('Time (ms)')  # x-axis label
        ax.set_title(f'Chunk Size: {chunk_ms}ms')  # subplot title
        ax.legend(loc='upper right', fontsize=8)  # show TTFA legend
        ax.grid(axis='x', alpha=0.3)  # vertical grid
    
    plt.suptitle('VoxServe Streaming Pipeline: Chunk Size vs TTFA', fontsize=12)
    plt.tight_layout()  # prevent overlap
    plt.show()  # render
    
    # Summary: TTFA vs chunk size tradeoff
    print('Chunk Size -> Time-to-First-Audio:')
    for cs, ttfa in zip(chunk_sizes_ms, ttfa_results):
        utilization = thinker_time_per_chunk * (cs/200) / (thinker_time_per_chunk * (cs/200) + 5) * 100
        print(f'  {cs:4d}ms chunks: TTFA = {ttfa:5.1f}ms, Thinker utilization ~ {utilization:.0f}%')
    return chunk_sizes_ms, ttfa_results

# Execute experiment 3
exp3_chunks, exp3_ttfa = experiment_3_streaming_pipeline()

## Experiment 4: Walk Graph Routing Efficiency

Compare flat (all components always run) vs graph-based scheduling (only traverse needed nodes). Measure compute savings from routing requests through minimal walks.

In [ ]:
def experiment_4_walk_graph_routing():
    """Compare flat pipeline vs walk-graph scheduling efficiency."""
    np.random.seed(42)  # reproducibility
    
    # Define available compute nodes and their costs (relative FLOPS units)
    node_costs = {
        'vision_enc': 15,   # ViT encoder cost
        'audio_enc': 10,    # audio encoder cost
        'prefill': 20,      # LLM prefill cost
        'decode': 50,       # autoregressive decode cost (dominant)
        'audio_codec': 8,   # speech synthesis cost
        'diffusion': 40,    # image generation cost
    }
    
    # Define walks (request types and their required nodes)
    walks = {
        'text_only': ['prefill', 'decode'],
        'image_qa': ['vision_enc', 'prefill', 'decode'],
        'speech_out': ['prefill', 'decode', 'audio_codec'],
        'image_to_speech': ['vision_enc', 'prefill', 'decode', 'audio_codec'],
        'image_gen': ['prefill', 'decode', 'diffusion'],
        'full_omni': ['vision_enc', 'audio_enc', 'prefill', 'decode', 'audio_codec', 'diffusion'],
    }
    
    # All nodes cost for flat pipeline (always runs everything)
    flat_cost = sum(node_costs.values())  # total if all nodes always execute
    
    # Compute cost per walk (graph-based: only needed nodes)
    walk_costs = {}  # store per-walk costs
    for walk_name, nodes in walks.items():
        walk_costs[walk_name] = sum(node_costs[n] for n in nodes)  # sum only required nodes
    
    # Simulate a realistic request mix (1000 requests)
    n_requests = 1000  # total simulated requests
    # Probability distribution over walk types
    walk_probs = [0.40, 0.25, 0.15, 0.10, 0.07, 0.03]  # text-heavy workload
    walk_names_list = list(walks.keys())  # ordered walk names
    
    # Sample request types
    request_indices = np.random.choice(len(walk_names_list), size=n_requests, p=walk_probs)
    
    # Compute total cost under each scheduling strategy
    total_flat = flat_cost * n_requests  # flat: always run all nodes
    total_graph = sum(walk_costs[walk_names_list[i]] for i in request_indices)  # graph: run only needed
    savings_pct = (total_flat - total_graph) / total_flat * 100  # percentage saved
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Left: per-walk cost comparison vs flat
    walk_labels = [n.replace('_', '\n') for n in walk_names_list]  # formatted labels
    graph_costs_arr = np.array([walk_costs[w] for w in walk_names_list])  # graph costs
    flat_costs_arr = np.full(len(walk_names_list), flat_cost)  # flat cost (constant)
    
    x_pos = np.arange(len(walk_names_list))  # x positions
    bw = 0.35  # bar width
    
    ax1.bar(x_pos - bw/2, flat_costs_arr, bw, label='Flat (all nodes)',
            color='#ffe4e6', edgecolor='#000')  # flat cost bars
    ax1.bar(x_pos + bw/2, graph_costs_arr, bw, label='Graph (walk only)',
            color='#dcfce7', edgecolor='#000')  # graph cost bars
    
    ax1.set_xlabel('Request Type (Walk)')  # x-axis
    ax1.set_ylabel('Compute Cost (FLOPS units)')  # y-axis
    ax1.set_title('Per-Request Cost: Flat vs Graph Scheduling')  # title
    ax1.set_xticks(x_pos)  # tick positions
    ax1.set_xticklabels(walk_labels, fontsize=8)  # tick labels
    ax1.legend()  # show legend
    ax1.grid(axis='y', alpha=0.3)  # grid
    
    # Right: cumulative cost over 1000 requests
    cumulative_flat = np.cumsum(np.full(n_requests, flat_cost))  # flat cumulative
    cumulative_graph = np.cumsum([walk_costs[walk_names_list[i]] for i in request_indices])  # graph cumulative
    
    ax2.plot(range(n_requests), cumulative_flat / 1000, color='#991b1b',
             label='Flat scheduling', linewidth=2)  # flat line
    ax2.plot(range(n_requests), cumulative_graph / 1000, color='#166534',
             label='Graph scheduling', linewidth=2)  # graph line
    ax2.fill_between(range(n_requests), cumulative_graph/1000, cumulative_flat/1000,
                     alpha=0.15, color='#dcfce7')  # shade savings area
    
    ax2.set_xlabel('Request Count')  # x-axis
    ax2.set_ylabel('Cumulative Compute (KFLOPS units)')  # y-axis
    ax2.set_title(f'Cumulative Savings: {savings_pct:.1f}% reduction')  # title with result
    ax2.legend()  # show legend
    ax2.grid(alpha=0.3)  # grid
    
    plt.tight_layout()  # prevent overlap
    plt.show()  # render
    
    # Print summary
    print(f'Total flat cost:  {total_flat:,} FLOPS units')  # total flat
    print(f'Total graph cost: {total_graph:,} FLOPS units')  # total graph
    print(f'Savings: {savings_pct:.1f}% compute reduction with walk-graph routing')  # savings
    print(f'\nPer-walk savings vs flat ({flat_cost} units):')
    for wn in walk_names_list:
        saved = (flat_cost - walk_costs[wn]) / flat_cost * 100  # per-walk savings
        print(f'  {wn:20s}: {walk_costs[wn]:3d} units ({saved:.0f}% saved)')  # print each
    return total_flat, total_graph, savings_pct

# Execute experiment 4
exp4_flat, exp4_graph, exp4_savings = experiment_4_walk_graph_routing()

## Key Takeaways

1. **Multimodal models are 2-12x more compute-intensive** than text-only, with heterogeneous stage profiles
2. **EPD disaggregation reduces latency 5-25%** for multimodal requests by eliminating pipeline bubbles
3. **Chunked streaming (200ms)** achieves sub-300ms TTFA while maintaining high GPU utilization
4. **Walk-graph routing saves 30-50%+ compute** by only running the nodes each request actually needs